# Solving Quantum Linear Systems Using Non-Hybrid Solutions

The main task of quantum linear system solving is to solve
$
Ax=b,
$
where $A$ is a matrix and $b$ is the known vector. In the classical case, we want to find the vector $x$ itself. In the quantum case, the goal is usually to prepare the quantum state
$|x\rangle$ , which is proportional to $ A^{-1}|b\rangle.$

This means that a quantum algorithm usually does not return the full solution in classical form. Instead, it prepares a state from which information about the solution can be extracted, for example, to estimate some quantity.

The most important parameters for solving linear systems using quantum algorithms are:

$
N -
$
the system size, that is, the dimension of the matrix. If the matrix is $N \times N$, then the system has $N$ equations and $N$ unknowns.

$
s -
$
the sparsity of the matrix. This describes how few nonzero elements the matrix has. A matrix is $s$-sparse if each row or column contains at most $s$ nonzero elements.

$
\kappa -
$
the condition number. This shows how “difficult” or unstable the linear system is. If $\kappa$ is large, then small errors in the data can cause larger errors in the solution, and the system is harder to solve.

$
\varepsilon -
$
the allowed error, or the desired precision. The smaller $\varepsilon$ is, the more accurate a result we want to obtain, but this usually also means higher computational cost.

Quantum algorithms for this problem are interesting because, in some cases, their complexity depends on $N$ only as O(poly(logN)). However, this is advantageous only if the matrix has suitable structure and it is not necessary to read out the entire solution vector in classical form.

## HHL

HHL is the most famous quantum linear systems algorithm. Its main idea is to use the eigenvalues of the matrix.  

- $A$ is an $N \times N$ matrix that describes the linear system,
- $|u_j\rangle$ are the eigenvectors of the matrix $A$ (an orthonormal basis),
- $\lambda_j$ are the corresponding eigenvalues,
- $|b\rangle$ is a normalized quantum state encoding the right-hand-side vector $b$,
- $\beta_j = \langle u_j | b \rangle$ are the projection coefficients (complex numbers) that determine how much of each eigenvector is contained in $|b\rangle$.

$$
A|u_j\rangle=\lambda_j|u_j\rangle,
$$

$$
|b\rangle=\sum_j \beta_j |u_j\rangle,
$$
so $|b\rangle$ is expressed in the eigenvector basis, and $\beta_j$ are the coefficients of this decomposition.

$$
A^{-1}|b\rangle=\sum_j \beta_j \frac{1}{\lambda_j}|u_j\rangle.
$$
This means that, in this basis, matrix inversion acts by multiplying each component $\beta_j$ by $\frac{1}{\lambda_j}$, provided that $\lambda_j \neq 0$. As a result, the solution vector is obtained in the form of a quantum state.

HHL tries to introduce the factor
$
1/\lambda_j.
$

The main steps of HHL are:
- prepare the state $|b\rangle$,
- use quantum phase estimation,
- obtain the eigenvalues $\lambda_j$,
- introduce the factor $1/\lambda_j$ using controlled rotations,
- obtain a state proportional to $A^{-1}|b\rangle$.

HHL was the first major algorithm for this problem. However, it is not always the most efficient option, because its complexity is rather sensitive to the condition number $\kappa$ and the error $\varepsilon$.

$
O(s^2\kappa^2\log N/\varepsilon).
$

In newer versions of Qiskit, a high-level implementation of the HHL algorithm is no longer available, so it must be implemented at a lower level.

## Low-Level HLL example


In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate, RYGate, UnitaryGate
from qiskit.quantum_info import Statevector

# Classical solution
A = np.array([
    [1, 0],
    [0, 2]
], dtype=float)

b = np.array([
    1/np.sqrt(2),
    1/np.sqrt(2)
], dtype=float)

x_classical = np.linalg.solve(A, b)  # solves Ax=b in the classical way

# HHL finally gives a normalized quantum state,
# so we also normalize the classical solution
x_classical_norm = x_classical / np.linalg.norm(x_classical)

print("=== Classical solution ===")
print("x =", np.round(x_classical, 6))
print("Normalized |x> =", np.round(x_classical_norm, 6))

# The matrix is diagonal: A = diag(1, 2), therefore
# |0> is an eigenstate with lambda = 1
# |1> is an eigenstate with lambda = 2

t = np.pi / 2

# U = exp(iAt)
# Since A is diagonal, U is also diagonal
U = np.diag([
    np.exp(1j * 1 * t),   # for lambda = 1
    np.exp(1j * 2 * t)    # for lambda = 2
])

# U^2
U2 = np.linalg.matrix_power(U, 2)

# Convert the matrices into Qiskit gate objects
U_gate = UnitaryGate(U, label="U")
U2_gate = UnitaryGate(U2, label="U^2")

# q0 = system qubit - it contains |b> and at the end the solution state |x>
# q1 = phase register LSB
# q2 = phase register MSB
# q3 = ancilla qubit - in the HHL algorithm this is used
# to introduce the factor 1/lambda into the amplitudes

qc = QuantumCircuit(4)

# |b>
qc.h(0)

# For QPE, the phase register is first put into superposition
qc.h(1)
qc.h(2)

qc.append(U_gate.control(1), [1, 0])
qc.append(U2_gate.control(1), [2, 0])

# inverse QFT on the phase register
qc.append(QFTGate(2).inverse(), [1, 2])

# HHL wants the ancilla amplitude on |1> to be proportional to C / lambda,
# where C is a constant. Here we choose C = 1.
# if lambda = 1, then the amplitude on ancilla |1> should be 1
# if lambda = 2, then it should be 1/2

# We need to find the corresponding Ry angle.
# Ry(theta)|0> = cos(theta/2)|0> + sin(theta/2)|1>

# If we want amplitude alpha on |1>, then we need sin(theta/2) = alpha
# lambda = 1, alpha = 1, theta = pi
# lambda = 2, alpha = 1/2, theta = pi/3

theta_l1 = np.pi
theta_l2 = np.pi / 3

# ry_l1 is activated only when the phase register = |01>
# ry_l2 is activated only when the phase register = |10>

# if QPE found lambda=1, we apply theta=pi
# if QPE found lambda=2, we apply theta=pi/3

ry_l1 = RYGate(theta_l1).control(2, ctrl_state='01')
ry_l2 = RYGate(theta_l2).control(2, ctrl_state='10')

# The control qubits are q1 and q2, the target is q3
qc.append(ry_l1, [1, 2, 3])
qc.append(ry_l2, [1, 2, 3])

# Inverse QPE
qc.append(QFTGate(2), [1, 2])
qc.append(U2_gate.control(1).inverse(), [2, 0])
qc.append(U_gate.control(1).inverse(), [1, 0])
qc.h(2)
qc.h(1)

# The Statevector contains the final state of all 4 qubits.
# But we do not want to read all 16 amplitude elements.
# We are only interested in the "successful branch".

sv = Statevector.from_instruction(qc)
vec = sv.data

# We need the ancilla qubit = 1, that is q3 = 1,
# and the phase register = 00, that is q2 = 0, q1 = 0
# Then the system qubit q0 can be either 0 or 1

# Therefore the two required basis states are:
# |1 0 0 0>, index 8
# |1 0 0 1>, index 9

# These two numbers together give the system state
# in the successful HHL branch.

solution_branch = np.array([vec[8], vec[9]], dtype=complex)

# This is the probability that ancilla = 1 and phase register = 00,
# that is, the successful HHL outcome.

success_prob = np.sum(np.abs(solution_branch)**2)

# HHL gives a branch that is proportional to the solution.
# To compare it with the classical |x>, we normalize it.
x_hhl = solution_branch / np.linalg.norm(solution_branch)

# Sometimes very small imaginary parts remain,
# which arise only from numerical errors.
x_hhl_real = np.real_if_close(x_hhl, tol=1000)

print("\n=== HHL result ===")
print("Successful branch probability =", round(float(success_prob), 6))
print("HHL obtained |x> =", np.round(x_hhl_real, 6))

print("\n=== Comparison with the classical answer ===")
print("Classical |x> =", np.round(x_classical_norm, 6))
print("HHL       |x> =", np.round(x_hhl_real, 6))

difference = np.linalg.norm(x_classical_norm - x_hhl_real)
print("Difference =", difference)

if difference < 1e-10:
    print("\nConclusion: the HHL result matches the classical solution.")
else:
    print("\nConclusion: the results do not match exactly, the circuit should be checked.")

print("\n=== HHL circuit ===")
print(qc.draw())

=== Classical solution ===
x = [0.707107 0.353553]
Normalized |x> = [0.894427 0.447214]

=== HHL result ===
Successful branch probability = 0.625
HHL obtained |x> = [0.894427 0.447214]

=== Comparison with the classical answer ===
Classical |x> = [0.894427 0.447214]
HHL       |x> = [0.894427 0.447214]
Difference = 2.7755575615628914e-16

Conclusion: the HHL result matches the classical solution.

=== HHL circuit ===
     ┌───┐┌───┐┌─────┐                                       ┌─────────┐»
q_0: ┤ H ├┤ U ├┤ U^2 ├───────────────────────────────────────┤ Unitary ├»
     ├───┤└─┬─┘└──┬──┘┌─────────┐                    ┌──────┐└────┬────┘»
q_1: ┤ H ├──■─────┼───┤0        ├────■─────────o─────┤0     ├─────┼─────»
     ├───┤        │   │  qft_dg │    │         │     │  Qft │     │     »
q_2: ┤ H ├────────■───┤1        ├────o─────────■─────┤1     ├─────■─────»
     └───┘            └─────────┘┌───┴───┐┌────┴────┐└──────┘           »
q_3: ────────────────────────────┤ Ry(π) ├┤ Ry(π/3) ├─────────

- - -
## Block-Encoding and QSVT

This is a more modern non-hybrid approach. Its idea is to embed the matrix $A$ into a larger unitary operator and then implement a function that acts on the eigenvalues similarly to
$$
1/\lambda.
$$

### The idea of block-encoding

Instead of directly implementing $A$, a unitary operator $U_A$ is constructed that contains the normalized matrix $A/\alpha$ as a block:
$$
\langle 0| U_A |0\rangle = \frac{A}{\alpha},
$$
where $\alpha \ge \|A\|$.

This means that if the system is in the state $|0\rangle \otimes |b\rangle$, then after applying $U_A$, part of the amplitude contains information about
$$
\frac{A}{\alpha}|b\rangle.
$$

If
$$
A|u_j\rangle = \lambda_j |u_j\rangle,
$$
then in the block-encoding case, one effectively manipulates the normalized values
$$
\tilde{\lambda}_j = \lambda_j / \alpha.
$$

### QSVT (Quantum Singular Value Transformation)

QSVT allows us to implement a function $f(x)$ on the eigenvalues of a matrix.

If
$$
|b\rangle = \sum_j \beta_j |u_j\rangle,
$$
then after the QSVT transformation we obtain:
$$
f(A/\alpha)|b\rangle = \sum_j \beta_j f(\lambda_j/\alpha) |u_j\rangle.
$$

To solve the linear system, we choose the function
$$
f(x) \approx \frac{1}{x},
$$
thus:
$$
f(A/\alpha)|b\rangle \approx \sum_j \beta_j \frac{\alpha}{\lambda_j} |u_j\rangle.
$$

Therefore:
$$
|x\rangle \propto A^{-1}|b\rangle.
$$

### Complexity:
$$
O(s \kappa\, \mathrm{polylog}(N,\kappa,1/\varepsilon)).
$$

In [3]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import RYGate
from qiskit.quantum_info import Statevector, Operator

# Linear system A x = b
A = np.array([
    [1, 0],
    [0, 2]
], dtype=float)

b = np.array([
    1 / np.sqrt(2),
    1 / np.sqrt(2)
], dtype=float)

# We choose alpha = 2, because it is the largest eigenvalue of the matrix A,
# ||A|| = 2. For the block-encoding to be correct, we need ||A/alpha|| ≤ 1.
alpha = 2.0

# For this diagonal matrix, the eigenvalues are the diagonal elements
eigvals = np.diag(A)

# Classical solution
x_classical = np.linalg.solve(A, b)
x_classical_norm = x_classical / np.linalg.norm(x_classical)  # normalize the result

print("=== Classical solution ===")
print("x =", np.round(x_classical, 6))
print("Normalized |x> =", np.round(x_classical_norm, 6))

# q0 = ancilla qubit
# q1 = system qubit

# We want <0|U_A|0> = A/alpha
# Therefore, for each eigenvalue λ_j, the corresponding coefficient is λ_j / alpha
be_coef = eigvals / alpha

# For the |0> state we need the coefficient λ_0/alpha = 1/2
# RY(theta)|0> gives the amplitude on |0> equal to cos(theta/2),
# so we obtain the angle from cos(theta/2) = λ_0/alpha
theta_be = 2 * np.arccos(be_coef[0])

U_A_circuit = QuantumCircuit(2, name="U_A")  # circuit representing U_A
c0_ry = RYGate(theta_be).control(1, ctrl_state='0')  # activated when q1 is 0
U_A_circuit.append(c0_ry, [1, 0])

print("\n=== Block-encoding circuit U_A ===")
print(U_A_circuit.draw())

U_A = Operator(U_A_circuit).data  # converts the quantum circuit into its matrix form
print("U_A matrix =")
print(np.round(U_A, 6))

print("\nThe upper-left block with respect to the ancilla=0 branch gives A/alpha.")
print("A/alpha =")
print(np.round(A / alpha, 6))

# In the inversion case, we need the coefficients 1/λ_j
inv_coef = 1 / eigvals

# For the |1> state we need the coefficient 1/λ_1 = 1/2
# Again, we use cos(theta/2) = desired coefficient
theta_inv = 2 * np.arccos(inv_coef[1])

U_inv_circuit = QuantumCircuit(2, name="U_inv")
c1_ry = RYGate(theta_inv).control(1, ctrl_state='1')
U_inv_circuit.append(c1_ry, [1, 0])

print("\n=== 'QSVT inverse' circuit for this example ===")
print(U_inv_circuit.draw())

U_inv = Operator(U_inv_circuit).data
print("U_inv matrix =")
print(np.round(U_inv, 6))

qc = QuantumCircuit(2)
qc.h(1)  # Prepare |b> on the system qubit

# Apply the transformed operator
qc.compose(U_inv_circuit, inplace=True)

print("\n=== Final circuit ===")
print(qc.draw())

sv = Statevector.from_instruction(qc)  # computes the final quantum state of the circuit
vec = sv.data  # takes the final state's amplitudes as a NumPy array

# We are only interested in the cases where ancilla q0 = 0,
# so we take the amplitudes from the states |00> and |10>
solution_branch = np.array([vec[0], vec[2]], dtype=complex)

success_prob = float(np.sum(np.abs(solution_branch) ** 2))
x_qsvt = solution_branch / np.linalg.norm(solution_branch)
x_qsvt = np.real_if_close(x_qsvt, tol=1000)

print("\n=== 5) Result ===")
print("Successful branch probability =", round(success_prob, 6))
print("Block-encoding/QSVT obtained |x> =", np.round(x_qsvt, 6))

print("\n=== 6) Comparison with the classical answer ===")
print("Classical |x> =", np.round(x_classical_norm, 6))
print("Quantum   |x> =", np.round(x_qsvt, 6))

difference = np.linalg.norm(x_classical_norm - x_qsvt)
print("Difference =", difference)

if difference < 1e-10:
    print("\nConclusion: the result matches the classical normalized solution.")
else:
    print("\nConclusion: the results do not match exactly.")

=== Classical solution ===
x = [0.707107 0.353553]
Normalized |x> = [0.894427 0.447214]

=== Block-encoding circuit U_A ===
     ┌──────────┐
q_0: ┤ Ry(2π/3) ├
     └────┬─────┘
q_1: ─────o──────
                 
U_A matrix =
[[ 0.5     +0.j -0.866025+0.j  0.      +0.j  0.      +0.j]
 [ 0.866025+0.j  0.5     +0.j  0.      +0.j  0.      +0.j]
 [ 0.      +0.j  0.      +0.j  1.      +0.j  0.      +0.j]
 [ 0.      +0.j  0.      +0.j  0.      +0.j  1.      +0.j]]

The upper-left block with respect to the ancilla=0 branch gives A/alpha.
A/alpha =
[[0.5 0. ]
 [0.  1. ]]

=== 'QSVT inverse' circuit for this example ===
     ┌──────────┐
q_0: ┤ Ry(2π/3) ├
     └────┬─────┘
q_1: ─────■──────
                 
U_inv matrix =
[[ 1.      +0.j  0.      +0.j  0.      +0.j  0.      +0.j]
 [ 0.      +0.j  1.      +0.j  0.      +0.j  0.      +0.j]
 [ 0.      +0.j  0.      +0.j  0.5     +0.j -0.866025+0.j]
 [ 0.      +0.j  0.      +0.j  0.866025+0.j  0.5     +0.j]]

=== Final circuit ===
          ┌────

## Other Quantum Linear System Solving Methods

In addition to the HHL and block-encoding/QSVT approaches, the literature also mentions the following methods:

- Adiabatic quantum linear system methods
- Discrete adiabatic methods
- LCU (Linear Combination of Unitaries) methods
- Chebyshev polynomial methods
- Krylov subspace methods
- Variational quantum linear solvers (VQLS)
- Hybrid quantum-classical methods
- Shadow quantum linear solvers
- Neural network / variational neural approaches

More detailed information about these methods can be found in the reference materials.

## References

This material is based on:  
Emergent Mind, *Quantum Linear Solvers (QLSS).*  
https://www.emergentmind.com/topics/quantum-linear-solvers-qlss

Emergent Mind, *Quantum Singular Value Transformation (QSVT).*  
https://www.emergentmind.com/topics/quantum-singular-value-transformation-qsvt